<a href="https://colab.research.google.com/github/nisha-muthurajan/LCA/blob/main/lca_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn==1.6.1 joblib numpy


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import joblib


In [ ]:
df_met = pd.read_csv("/content/metallurgical_data.csv")
df_min = pd.read_csv("/content/mining_process_data.csv")

print(df_met.head())
print(df_min.head())


FileNotFoundError: [Errno 2] No such file or directory: '/content/metallurgical_data.csv'

In [ ]:
df_met.columns = df_met.columns.str.lower().str.strip()
df_min.columns = df_min.columns.str.lower().str.strip()


In [ ]:
df_met_rename_map = {
    'ore_type': 'material_type',
    'extraction_method': 'process_stage',
    'energy_consumption_kwh': 'energy_consumption',
    'co2_emissions_tons': 'co2_emission',
    'water_recycled_percent': 'water_usage',
    'waste_to_byproduct_ratio': 'waste_generated'
}

df_min_rename_map = {
    'energy_kwh': 'energy_consumption',
    'co2_emissions_kg': 'co2_emission',
    'water_usage_liters': 'water_usage',
    'waste_generated_kg': 'waste_generated'
}

# Apply renaming to both dataframes
df_met = df_met.rename(columns=df_met_rename_map)
df_min = df_min.rename(columns=df_min_rename_map)

# Define the final common columns that are present in both dataframes after renaming
# Note: 'material_type' and 'process_stage' from the original common_cols are not
# directly present in df_min even after renaming, so they are excluded for consistency across both.
final_common_cols = [
    'energy_consumption',
    'water_usage',
    'co2_emission',
    'waste_generated'
]

df_met = df_met[final_common_cols]
df_min = df_min[final_common_cols]

In [ ]:
df = pd.concat([df_met, df_min], ignore_index=True)
print("Combined dataset shape:", df.shape)

Combined dataset shape: (35, 4)


In [ ]:
df['energy_consumption'].fillna(df['energy_consumption'].median(), inplace=True)
df['water_usage'].fillna(df['water_usage'].median(), inplace=True)
df['co2_emission'].fillna(df['co2_emission'].median(), inplace=True)
df['waste_generated'].fillna(df['waste_generated'].median(), inplace=True)


/tmp/ipython-input-3308255889.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['energy_consumption'].fillna(df['energy_consumption'].median(), inplace=True)
/tmp/ipython-input-3308255889.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].meth

In [ ]:
df['sustainability_score'] = (
    1 / (df['co2_emission'] + df['energy_consumption'] + df['waste_generated'])
)

# Normalize score
df['sustainability_score'] = (
    df['sustainability_score'] - df['sustainability_score'].min()
) / (
    df['sustainability_score'].max() - df['sustainability_score'].min()
)


In [ ]:
X = df.drop('sustainability_score', axis=1)
y = df['sustainability_score']

categorical_features = []
numerical_features = [
    'energy_consumption',
    'water_usage',
    'co2_emission',
    'waste_generated'
]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ]
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])


In [ ]:
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  []),
                                                 ('num', StandardScaler(),
                                                  ['energy_consumption',
                                                   'water_usage',
                                                   'co2_emission',
                                                   'waste_generated'])])),
                ('model',
                 RandomForestRegressor(n_estimators=300, random_state=42))])

In [ ]:
r2 = pipeline.score(X_test, y_test)
print(f"Model R² Score: {r2:.3f}")


Model R² Score: 0.985


In [ ]:
print("Train R2:", pipeline.score(X_train, y_train))
print("Test R2:", pipeline.score(X_test, y_test))


Train R2: 0.9964288751433441
Test R2: 0.9854497508308528


In [ ]:
joblib.dump(pipeline, "ai_circularity_recommendation_model.pkl")
print("Model saved successfully!")


Model saved successfully!
